# Who picks the tool? A walkthrough

This notebook recomputes every number in the MeasureCamp talk from the committed runs, and shows you how to
make new calls if you have keys. It runs without any API key.

**The question.** An AI support assistant gets a customer message and 91 tools (90 copied verbatim from
official MCP servers, plus `hand_off_to_human`). Which tool should it call first? We compare four selectors:

- **Jev** (TypeSafe System One): a small model that answers a `choice` question with a probability per option.
- **DeepSeek V4.1 Flash, Claude Haiku 5.5, Claude Sonnet 5.5**, each in two modes: as a *router* (told to
  reply with one tool's name) and with *native tool calling* (the tools in the API's `tools` parameter).

Each LLM run exists under two prompts (*base* and *policy*), and everything is scored under two label
versions (*v1*, *v2*). The point of the talk is that the ranking depends on those choices, so you should see them.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import toolpick as tp

pd.set_option("display.max_colwidth", 90)
print(len(tp.requests()), "messages ·", len(tp.tools()), "tools ·", len(tp.available_runs()), "runs:", tp.available_runs())

600 messages · 91 tools · 14 runs: ['jev', 'jev_policy', 'native_deepseek', 'native_deepseek_policy', 'native_haiku', 'native_haiku_policy', 'native_sonnet', 'native_sonnet_policy', 'router_deepseek', 'router_deepseek_policy', 'router_haiku', 'router_haiku_policy', 'router_sonnet', 'router_sonnet_policy']


## 1. The data

600 messages, a fixed sample of Bitext's customer-support dataset. The text is template-generated with deliberate
typos; the `intent` is Bitext's label, which we map to tools.

In [2]:
from dataclasses import asdict

reqs = pd.DataFrame([asdict(r) for r in tp.requests()])
display(reqs.sample(8, random_state=1))
reqs["intent"].value_counts().to_frame("messages").T

,id,intent,text
446,R0446,registration_problems,I cannot inform of an error with sign-up
404,R0404,delete_account,i dont know what to do to delete my platinum account
509,R0509,check_cancellation_fee,can you help me seeing the termination fees?
455,R0455,contact_human_agent,I want help to speak to somebody
201,R0201,edit_account,need to modify the user info help me
23,R0023,change_order,wanna delete some items from purchase ORD-436202
415,R0415,cancel_order,i need help cancelling purchase ORD-407472
548,R0548,track_order,where to see the bloody ETA of the order ORD-473003


intent,contact_human_agent,delivery_period,payment_issue,check_cancellation_fee,contact_customer_service,place_order,check_refund_policy,review,newsletter_subscription,create_account,...,registration_problems,check_payment_methods,get_refund,track_refund,recover_password,edit_account,delete_account,cancel_order,delivery_options,change_shipping_address
messages,32,29,29,29,28,27,26,26,26,25,...,21,20,19,18,17,15,15,13,13,12


The tools. Every name, description and schema was taken from the vendor: by running the official package and
calling `tools/list`, by calling a live public Shopify store, or from official docs (Intercom). `data/REPORT.md`
has the provenance of each server. Note how uneven the prompt budget is.

In [3]:
cat = pd.DataFrame(tp.catalog())[["qualified_name", "server", "description", "exactness"]]
display(cat.groupby("server").size().to_frame("tools").T)
tok = tp.tool_tokens_by_server()
print({k: f"{v / sum(tok.values()):.0%}" for k, v in sorted(tok.items(), key=lambda kv: -kv[1])}, "of the tool JSON")
cat[cat["server"] == "shopify_customer_account"]

server,github,intercom,notion,shopify_customer_account,shopify_storefront,shopify_ucp,stripe
tools,10,14,24,4,1,15,22


{'notion': '41%', 'shopify_ucp': '38%', 'stripe': '10%', 'github': '8%', 'intercom': '1%', 'shopify_customer_account': '1%', 'shopify_storefront': '1%'} of the tool JSON


,qualified_name,server,description,exactness
16,shopify_customer_account.get_most_recent_order_status,shopify_customer_account,Gets the status of the customer's most recent order.,exact (tools/list)
17,shopify_customer_account.get_order_status,shopify_customer_account,Fetches a specific order by order number for a logged-in customer and returns status d...,exact (tools/list)
18,shopify_customer_account.get_store_credit_balances,shopify_customer_account,Fetches the customer's store credit balances if they have any.,exact (tools/list)
19,shopify_customer_account.request_return,shopify_customer_account,Allows a customer to request a return for an order.\nEither an order_id or an order_nu...,exact (tools/list)


## 2. The labels

Per intent: `acceptable` tools (a clean fit), `partial` tools (a defensible first step that doesn't finish the job).
**15 of the 27 intents have no real tool**; there the only acceptable answer is `hand_off_to_human`.

`v2` is the revised version that also accepts the readings the LLMs made consistently: an FAQ search for a
how-to question on a gap intent, a hand-off on `payment_issue`, the refund-policy FAQ on `get_refund`. Both
versions, with every override and its reason, live in `src/toolpick/data.py`.

In [4]:
lab = tp.labels("v1")
pd.DataFrame([{"intent": k, "no tool": v["gap"], "acceptable": ", ".join(t.split(".")[-1] for t in v["acceptable"]),
               "partial": ", ".join(t.split(".")[-1] for t in v["partial"]), "override": v["override"] or ""}
              for k, v in lab.items()]).sort_values(["no tool", "intent"])

,intent,no tool,acceptable,partial,override
3,check_cancellation_fee,False,search_shop_policies_and_faqs,,
4,check_invoice,False,list_invoices,get_order,
5,check_payment_methods,False,search_shop_policies_and_faqs,"get_checkout, create_checkout",
6,check_refund_policy,False,search_shop_policies_and_faqs,,
8,contact_customer_service,False,search_shop_policies_and_faqs,hand_off_to_human,Handing to a person is a defensible reading of 'contact customer service'.
12,delivery_options,False,search_shop_policies_and_faqs,"search_locations, create_checkout",
13,delivery_period,False,"search_shop_policies_and_faqs, get_order_status, get_most_recent_order_status, get_order",,"get_order is accepted for track_order, so it is accepted here too."
15,get_invoice,False,list_invoices,get_order,
16,get_refund,False,"request_return, create_refund",,
18,payment_issue,False,list_payment_intents,list_disputes,


## 3. Scoring one decision

Every row of a run has the selector's pick (or no pick, if a native model replied in text). `tp.outcome` turns a
pick into one of six outcomes. Only `right` counts as right; `wrong` and `bluffed` together are the *wrong-call
rate*, the number a customer would notice.

In [5]:
for pick in ["shopify_customer_account.get_order_status", "shopify_ucp.get_order", "hand_off_to_human",
             "shopify_storefront.search_shop_policies_and_faqs", None]:
    print(f"{str(pick):55s} track_order -> {tp.outcome(pick, lab['track_order']):12s}  cancel_order -> {tp.outcome(pick, lab['cancel_order'])}")

shopify_customer_account.get_order_status               track_order -> right         cancel_order -> bluffed
shopify_ucp.get_order                                   track_order -> right         cancel_order -> bluffed
hand_off_to_human                                       track_order -> handed_off    cancel_order -> right
shopify_storefront.search_shop_policies_and_faqs        track_order -> wrong         cancel_order -> defensible
None                                                    track_order -> asked         cancel_order -> asked


## 4. The headline table

`scored_rows` joins a run with the labels; `summarise` gives the rates. Change `version` to `"v2"` and watch the
ranking move.

In [6]:
def table(version="v1"):
    rows = []
    for name in tp.available_runs():
        s = tp.summarise(tp.scored_rows(name, version))
        rows.append({"run": name, "n": s["n"], "right": s["right"], "wrong_call": s["wrong_call"], "asked": s["asked"],
                     "handed_off": s["handoff"], "latency_s": s["latency_median_s"], "input_tokens": s["input_tokens_median"],
                     "$ per 1000": s["cost_per_1000"]})
    return pd.DataFrame(rows).set_index("run").sort_values("right", ascending=False)

(table("v1").style.format({"right": "{:.1%}", "wrong_call": "{:.1%}", "asked": "{:.1%}", "handed_off": "{:.1%}",
                           "latency_s": "{:.2f}", "input_tokens": "{:,.0f}", "$ per 1000": "{:.2f}"})
   .bar(subset=["right"], color="#8ab17d", vmin=0, vmax=1).bar(subset=["wrong_call"], color="#e9a3a3", vmin=0, vmax=1))

,n,right,wrong_call,asked,handed_off,latency_s,input_tokens,$ per 1000
run,,,,,,,,
router_sonnet_policy,300,85.0%,6.0%,0.0%,54.3%,1.54,"70,069",8.04
jev_policy,600,82.2%,1.0%,0.0%,67.8%,0.30,"7,061",0.30
router_deepseek_policy,600,78.3%,4.8%,0.0%,64.5%,0.78,"43,975",0.37
native_sonnet_policy,600,77.7%,6.8%,9.8%,50.8%,3.21,"76,015",9.07
jev,600,73.8%,2.3%,0.0%,73.5%,0.29,"6,921",0.29
router_haiku_policy,600,73.3%,15.0%,0.3%,50.2%,1.20,"70,068",0.78
native_haiku_policy,600,70.0%,6.0%,17.0%,47.8%,2.81,"76,014",0.84
router_deepseek,600,66.3%,16.0%,0.0%,55.0%,0.80,"43,831",0.36
router_sonnet,300,62.7%,24.0%,0.7%,40.7%,1.60,"69,858",8.10


In [7]:
table("v2")[["right", "wrong_call"]].style.format("{:.1%}")

,right,wrong_call
run,,
router_sonnet_policy,93.0%,3.3%
jev_policy,88.3%,0.0%
router_sonnet,87.0%,7.7%
router_deepseek_policy,85.2%,3.0%
native_sonnet_policy,81.8%,4.7%
router_deepseek,80.7%,6.5%
jev,79.2%,1.8%
router_haiku_policy,78.2%,13.5%
native_haiku_policy,74.8%,4.0%


## 5. Paired comparisons

Two accuracies on the same messages aren't independent, so we don't compare them as if they were. `paired`
counts the messages where only one of the two was right (an exact McNemar test) and bootstraps the difference.

In [8]:
pairs = []
for other in ["router_deepseek", "router_haiku", "router_sonnet", "native_deepseek", "native_haiku", "native_sonnet"]:
    for suffix in ["", "_policy"]:
        p = tp.paired("jev" + suffix, other + suffix)
        pairs.append({"jev vs": other + suffix, "n": p["n_pairs"], "jev": p["right_a"], "other": p["right_b"],
                      "only jev right": p["only_a"], "only other right": p["only_b"], "diff": p["diff"],
                      "95% CI": f"[{p['ci'][0]:+.3f}, {p['ci'][1]:+.3f}]", "p": p["p"]})
pd.DataFrame(pairs).style.format({"jev": "{:.1%}", "other": "{:.1%}", "diff": "{:+.1%}", "p": "{:.2g}"})

,jev vs,n,jev,other,only jev right,only other right,diff,95% CI,p
0,router_deepseek,600,73.8%,66.3%,89,44,+7.5%,"[+0.037, +0.112]",0.00012
1,router_deepseek_policy,600,82.2%,78.3%,48,25,+3.8%,"[+0.012, +0.067]",0.0095
2,router_haiku,600,73.8%,47.0%,203,42,+26.8%,"[+0.222, +0.315]",1.7e-26
3,router_haiku_policy,600,82.2%,73.3%,89,36,+8.8%,"[+0.053, +0.125]",2.4e-06
4,router_sonnet,300,72.3%,62.7%,60,31,+9.7%,"[+0.040, +0.157]",0.0031
5,router_sonnet_policy,300,80.3%,85.0%,15,29,-4.7%,"[-0.090, -0.003]",0.049
6,native_deepseek,600,73.8%,28.7%,304,33,+45.2%,"[+0.403, +0.498]",4.7e-56
7,native_deepseek_policy,600,82.2%,33.7%,310,19,+48.5%,"[+0.438, +0.530]",6.3e-69
8,native_haiku,600,73.8%,37.2%,253,33,+36.7%,"[+0.318, +0.412]",3.6e-43
9,native_haiku_policy,600,82.2%,70.0%,96,23,+12.2%,"[+0.088, +0.157]",8.5e-12


## 6. Error analysis

Three views. First: when a selector is wrong, what does it reach for? Most LLM wrong calls are the store's FAQ
search on a question no tool can answer, which is exactly what the one-paragraph routing policy fixes.

In [9]:
def wrong_calls(name):
    rows = pd.DataFrame(tp.scored_rows(name))
    w = rows[rows["wrong_call"]]
    return (w.assign(pick=w["pick"].str.split(".").str[-1]).groupby(["intent", "pick"]).size()
             .sort_values(ascending=False).head(10).to_frame("n"))

display(wrong_calls("router_sonnet"))
wrong_calls("router_sonnet_policy")

,,n
intent,pick,
get_refund,search_shop_policies_and_faqs,10
set_up_shipping_address,search_shop_policies_and_faqs,10
review,search_shop_policies_and_faqs,6
switch_account,search_shop_policies_and_faqs,6
get_invoice,search_shop_policies_and_faqs,6
check_invoice,get_order_status,5
newsletter_subscription,search_shop_policies_and_faqs,5
place_order,search_shop_policies_and_faqs,5
create_account,search_shop_policies_and_faqs,3


,,n
intent,pick,
get_refund,search_shop_policies_and_faqs,4
payment_issue,search_shop_policies_and_faqs,3
check_invoice,get_order_status,2
get_refund,get_most_recent_order_status,2
check_invoice,list_customers,2
switch_account,search_shop_policies_and_faqs,2
place_order,search_shop_policies_and_faqs,1
review,search_shop_policies_and_faqs,1
set_up_shipping_address,search_shop_policies_and_faqs,1


Second: Jev's errors. Jev almost never calls the wrong tool; it hands off where a tool could have done the job.
That is a miss too (the customer waits for a person), so it counts as wrong, but it is a different kind of miss,
and Jev's confidence lets you choose how often to take it.

In [10]:
jev = pd.DataFrame(tp.scored_rows("jev"))
display(jev[~jev["gap"]].groupby("outcome").size().to_frame("covered intents"))
pd.DataFrame(tp.gate_curve("jev")).style.format({"acted": "{:.1%}", "wrong_of_acted": "{:.1%}", "right": "{:.1%}"})

,covered intents
outcome,
defensible,12
handed_off,130
right,144
wrong,3


,t,acted,wrong_of_acted,right
0,0.000000,26.5%,8.8%,73.8%
1,0.300000,26.5%,8.8%,73.8%
2,0.400000,25.5%,8.5%,73.5%
3,0.500000,21.7%,6.9%,70.8%
4,0.600000,17.3%,4.8%,67.7%
5,0.700000,12.0%,0.0%,63.8%
6,0.800000,9.2%,0.0%,61.0%
7,0.900000,6.5%,0.0%,58.3%
8,0.950000,5.0%,0.0%,56.8%


Third: the native models often reply in text instead of calling a tool. On vague messages that's reasonable; the
router condition can't do it. Scoring the first turn only makes native accuracy a lower bound.

In [11]:
nat = pd.DataFrame(tp.scored_rows("native_sonnet"))
asked = nat[nat["outcome"] == "asked"]
print(f"{len(asked)} of {len(nat)} replied in text")
asked[["intent", "text", "reply"]].sample(5, random_state=2)

147 of 600 replied in text


,intent,text,reply
336,get_refund,i have got to request reimbirsements of money,"I can help with that. To get started, I need a few details:\n\n1. Your order number (f..."
512,set_up_shipping_address,I have a problem entering another delivery address,I can help with that. Could you tell me a bit more about what's going wrong?\n\n- Are ...
5,delivery_period,where could I see how soon can I expect the product ?,"I can check that for you. Could you give me your order number? Once I have it, I'll lo..."
137,switch_account,"I have got to use the gold account, can I get some help?","I'd be glad to help. I'm not sure what you mean by ""the gold account"" yet. Could you t..."
550,check_invoice,checking invoice from 4 purchases ago,"I can help with that. To find the right invoice, I need to know which customer account..."


## 7. Browse disagreements

Pick two runs and look at the messages where they differ. This is the most useful thing to do with any eval:
the disagreements tell you whether the labels, the prompt or the model is at fault.

In [12]:
a = pd.DataFrame(tp.scored_rows("jev")).set_index("id")
b = pd.DataFrame(tp.scored_rows("router_sonnet_policy")).set_index("id")
both = a.join(b[["pick", "outcome", "reply"]], rsuffix="_sonnet", how="inner")
diff = both[both["pick"].fillna("-") != both["pick_sonnet"].fillna("-")]
print(len(diff), "disagreements")
diff[["intent", "text", "pick", "outcome", "pick_sonnet", "outcome_sonnet"]].head(12)

89 disagreements


,intent,text,pick,outcome,pick_sonnet,outcome_sonnet
id,,,,,,
R0000,place_order,i do not know what to do to acquire some of ur article,hand_off_to_human,handed_off,shopify_ucp.search_catalog,right
R0008,place_order,I need assistance shopping some of your item,hand_off_to_human,handed_off,shopify_ucp.search_catalog,right
R0010,check_cancellation_fee,how to check the early termination penalty?,hand_off_to_human,handed_off,shopify_storefront.search_shop_policies_and_faqs,right
R0022,place_order,I want help to order an article,hand_off_to_human,handed_off,shopify_ucp.search_catalog,right
R0024,check_cancellation_fee,want help to check the early termination penalties,hand_off_to_human,handed_off,shopify_storefront.search_shop_policies_and_faqs,right
R0030,contact_human_agent,what do i have to do to speak with somebody,hand_off_to_human,right,shopify_storefront.search_shop_policies_and_faqs,defensible
R0032,track_order,i need assistance to locate purchase ORD-499272,shopify_ucp.get_order,right,shopify_customer_account.get_order_status,right
R0046,place_order,purchase several of your product,hand_off_to_human,handed_off,shopify_ucp.search_catalog,right
R0048,check_cancellation_fee,i need assistance checking the damn cancellation charge,shopify_storefront.search_shop_policies_and_faqs,right,hand_off_to_human,handed_off


## 8. Make your own calls (optional)

With `OPENROUTER_API_KEY` and/or `TYPESAFE_API_KEY` in `.env`, the scripts in `scripts/` run any arm on any
number of messages and append to `runs/`. A few messages cost cents:

```bash
uv run python scripts/run_router.py --arm jev --limit 5
uv run python scripts/run_native.py --arm haiku --limit 5 --policy
```

The cell below shows one Jev call end to end.

In [13]:
import os
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")
if os.environ.get("TYPESAFE_API_KEY"):
    from toolpick.jev import Jev
    from toolpick.prompts import jev_instructions

    criteria = {t["name"]: t["description"] for t in tp.tools()}
    out = Jev().choose("i wqant assistance checking purchase ORD-475206 status", jev_instructions(policy=False), criteria)
    a = out["answers"]["tool"]
    print(a["choice"], f"confidence {a['confidence']:.2f}", f"{out['latency_s']:.2f}s", f"${out['cost_usd']:.5f}")
    print(sorted(a["probabilities"].items(), key=lambda kv: -kv[1])[:4])
else:
    print("no TYPESAFE_API_KEY: skipping the live call")

no TYPESAFE_API_KEY: skipping the live call


## What to take away

1. Use real tool text, sets of acceptable answers, and an explicit "no tool" class.
2. Report the wrong-call rate next to accuracy. A hand-off and a bluff are not the same miss.
3. Compare selectors on the same messages, with a paired test.
4. Write the routing policy down and put it in the prompt; then you're measuring the model, not its guess at your policy.
5. Publish the labels. They decide the winner.